# Lesson 4 — Classification and Evaluation Metrics

Self-assessment. No code: every answer is a sentence or a short calculation.

Work through a question before opening its answer. If your answer differs from
the one given, the interesting part is *where* the two diverge — that is usually
one specific belief, and it is worth finding.

The numbers quoted throughout come from the lesson's notebooks: 8,000 disk
drives, 306 of which failed, a test set of 2,000 drives containing 76
failures.

## Part 1 — The model

**1. Linear regression is fitted to a 0/1 label. Why is the result not usable as a probability, and why can it not be repaired by fitting more carefully?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>A straight line is <b>unbounded</b> while a probability lives in [0, 1], so for extreme feature values the line must leave the interval.</li>
        <li>In the lesson's fit, <b>3,606 of 8,000 drives — 45% of the fleet — received a negative probability</b>, and the line crosses 1 at 32 reallocated sectors.</li>
        <li>No choice of slope and intercept fixes it: the problem is the <b>shape of the function</b>, not the values of its parameters. That is why the sigmoid is needed.</li>
    </ul>
    </p>
</details>

**2. Define the odds and the log-odds of an event, and explain why logistic regression models the log-odds rather than the probability directly.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Odds are <code>p / (1 - p)</code>; log-odds are <code>log(p / (1 - p))</code>, also called the <b>logit</b>.</li>
        <li>A probability is trapped in [0, 1]. Odds remove the upper bound but keep a floor at 0. The logarithm removes the floor too, leaving a quantity spanning all of ℝ.</li>
        <li>That is exactly the range a linear model can safely produce — so <b>logistic regression is a linear model of the log-odds</b>, and the sigmoid is simply that relationship inverted.</li>
    </ul>
    </p>
</details>

**3. The sigmoid is steep in the middle and flat at both ends. Argue that this is the correct behaviour rather than a mathematical accident.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Evidence has <b>diminishing returns</b>. Going from 0 to 4 reallocated sectors should change your assessment a great deal; going from 40 to 44 should barely register, because the drive was already condemned.</li>
        <li>The flat tails encode that. The steep middle is where the model is genuinely undecided and new evidence is most informative.</li>
        <li>Quantitatively, the derivative is <code>σ(z)(1 - σ(z))</code>, largest at z = 0 where it equals 1/4, decaying to zero at both ends.</li>
    </ul>
    </p>
</details>

**4. A fitted coefficient is 1.80 on a standardised feature. State precisely what that means, and state one thing it does <i>not</i> mean.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>It means: a drive one <b>standard deviation</b> higher on that feature has <code>exp(1.80) ≈ 6.0</code> times the <b>odds</b> of failing.</li>
        <li>It does <b>not</b> mean six times the <i>probability</i>. Odds ratios multiply odds, not probabilities.</li>
        <li>The two coincide only when probabilities are small. Odds of 9 (p = 0.90) multiplied by 5 gives odds of 45, a probability of 0.978 — the odds rose fivefold while the probability moved under eight points.</li>
    </ul>
    </p>
</details>

**5. The fitted intercept is −6.09. What does that number say about the fleet, and why does it explain the behaviour of every metric later in the lesson?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>It is the log-odds of failure for a drive with perfectly average telemetry: <code>σ(-6.09) = 0.0023</code>, a <b>0.2% chance</b> over thirty days.</li>
        <li>To reach an even bet the evidence must move the log-odds by more than six, which only badly degraded drives manage.</li>
        <li>So the model assigns small probabilities to nearly everything, and a threshold of 0.5 flags almost nothing. That is not timidity — it is <b>correctness about a rare event</b>, and it is the source of the low recall seen later.</li>
    </ul>
    </p>
</details>

**6. One column, <code>seek_error_rate</code>, was generated with a coefficient of exactly zero. What did the model estimate, and what is the honest limitation of this demonstration?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The model estimated about <b>0.02</b>, an odds multiplier of 1.03 — effectively nothing. It did not fall for a plausible-looking counter.</li>
        <li>The limitation: <b>we knew the answer in advance</b>. On real data no column is labelled as a decoy.</li>
        <li>Recognising a useless feature without being told requires the validation machinery of lesson 5 — and note that a coefficient near zero on <i>this</i> sample is evidence, not proof.</li>
    </ul>
    </p>
</details>

## Part 2 — The cost function

**7. Derive the binary cross-entropy from maximum likelihood, in outline. Why is the logarithm taken?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Each example contributes <code>p^y (1-p)^(1-y)</code> — the probability the model assigned to what actually happened. Independence makes the dataset's probability the <b>product</b> over examples.</li>
        <li>The logarithm turns the product into a sum, which differentiates term by term, and it does not move the maximum because log is strictly increasing.</li>
        <li>It is also a <b>numerical necessity</b>: with 8,000 factors below 1 the product is around 10⁻¹⁰⁰⁰, which a 64-bit float stores as exactly zero.</li>
        <li>Negating gives a cost to minimise: <code>J = -(1/m) Σ [ y log p + (1-y) log(1-p) ]</code>.</li>
    </ul>
    </p>
</details>

**8. A drive fails and the model predicted 0.0001. Compute both losses. Then explain why the difference in <i>gradient</i> matters more than the difference in cost.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Squared error: <code>(0.0001 - 1)² ≈ 0.9998</code>. Log loss: <code>-log(0.0001) ≈ 9.21</code>.</li>
        <li>Squared error is <b>capped near 1</b>, only four times what it charges for an honest 0.5. Log loss is unbounded.</li>
        <li>The gradients matter more because the optimiser uses them. With respect to the log-odds: squared error gives <code>2(p-1)·p(1-p)</code>, log loss gives <code>p - 1</code>.</li>
        <li>The factor <code>p(1-p)</code> <b>vanishes as p → 0</b> — exactly where the model is most confidently wrong. At p = 0.001 the squared-error gradient is about 0.002 against log loss's 0.999. <b>Squared error stops teaching precisely where teaching is most needed.</b></li>
    </ul>
    </p>
</details>

**9. The gradient of the log loss is <code>(1/m) Σ (ŷ - y)x</code>, identical in form to linear regression's. Is that a coincidence?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>No. Both are <b>generalised linear models</b> fitted by maximum likelihood — Gaussian noise for linear regression, Bernoulli outcomes for logistic regression.</li>
        <li>For the whole exponential family, the maximum-likelihood gradient takes this form.</li>
        <li>The practical consequence: the same gradient descent implementation, unchanged, trains both.</li>
    </ul>
    </p>
</details>

**10. Log loss is convex in the weights. Does it follow that logistic regression has a closed-form solution like the normal equation?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>No.</b> Convexity guarantees that any local minimum is global, so gradient descent cannot be trapped — it says nothing about solving in closed form.</li>
        <li>Setting the gradient to zero gives <code>Xᵀ(σ(Xw + b) - y) = 0</code>, which is <b>not linear in w</b> because the sigmoid is in the way, and has no algebraic solution.</li>
        <li>There is no normal equation for logistic regression. It is solved iteratively — which is why <code>LogisticRegression</code> has a <code>max_iter</code> argument and <code>LinearRegression</code> does not.</li>
    </ul>
    </p>
</details>

## Part 3 — The accuracy trap

**11. On the test set of 2,000 drives, a model that predicts "healthy" for everything scores 96.20% accuracy. Explain the arithmetic, and state the general rule.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>1,924 of the 2,000 drives are healthy, so answering "healthy" collects 1,924 correct answers for free: 1924/2000 = 96.20%.</li>
        <li>The general rule: with a positive rate π, the trivial always-negative classifier scores <b>1 - π</b>.</li>
        <li>At π = 0.038 that is 96.2%; in a fraud problem with π = 0.001 it is 99.9%. <b>The rarer the interesting class, the more impressive the useless model looks.</b></li>
    </ul>
    </p>
</details>

**12. The real model scores 97.70%. Why is that gap of 1.5 points not evidence that the real model is only slightly better?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Only 3.8% of the test set is available to compete over, so <b>accuracy is almost entirely determined by the majority class</b> and cannot move far.</li>
        <li>The number that separates the two models is invisible in accuracy: <b>0 failures caught against 43 of 76</b>.</li>
        <li>Accuracy is not wrong here — it is answering a question nobody asked.</li>
    </ul>
    </p>
</details>

**13. Draw the confusion matrix for our model at threshold 0.5 (TP 43, FP 13, FN 33, TN 1911). Which cell should be read first, and why?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Rows are truth and columns are prediction in scikit-learn's convention: <code>[[1911, 13], [33, 43]]</code>.</li>
        <li><b>Read the false-negative cell first</b> when misses are expensive: 33 drives failed and were called healthy.</li>
        <li>That cell carries the €2,600-per-event cost, and it is the one every single-number metric hides. Note that some sources transpose the matrix — always check the axes before drawing conclusions.</li>
    </ul>
    </p>
</details>

## Part 4 — Precision, recall, and what to report

**14. Define precision and recall by their denominators, and give the question each one answers.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Precision = TP / (TP + FP)</b>, dividing by everything we <i>flagged</i> — the column of the matrix. "When this model raises an alarm, how often is it right?" The technician's question.</li>
        <li><b>Recall = TP / (TP + FN)</b>, dividing by everything that <i>actually was positive</i> — the row. "Of the drives that were going to fail, how many did we find?" The operations manager's question.</li>
        <li>The hook: <b>precision is about the alarms, recall is about the failures.</b> When in doubt, find the denominator and ask which population it counts.</li>
    </ul>
    </p>
</details>

**15. Our model flagged 56 drives, of which 43 really failed, out of 76 failures in total. Compute precision, recall and F1.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Precision = 43/56 = <b>0.768</b>.</li>
        <li>Recall = 43/76 = <b>0.566</b>.</li>
        <li>F1 = 2(0.768)(0.566) / (0.768 + 0.566) = <b>0.652</b>.</li>
        <li>Say each as a sentence with counts in it: of 56 alarms, 43 were real; of 76 failures, we found 43.</li>
    </ul>
    </p>
</details>

**16. A model that flags every drive has recall 1.000 and precision 0.038. Compute both means and explain why F1 uses the harmonic one.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Arithmetic mean: (1.000 + 0.038)/2 = <b>0.519</b> — a respectable-looking score for a model that flags everything.</li>
        <li>Harmonic mean: 2(1.000)(0.038)/(1.038) = <b>0.073</b>.</li>
        <li>The harmonic mean is <b>dominated by the smaller value</b>. A model is only as good as its weaker side, and F1 refuses to let a perfect score on one axis buy a pass on the other.</li>
    </ul>
    </p>
</details>

**17. When would you use F2 rather than F1?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><code>F_β = (1 + β²)·precision·recall / (β²·precision + recall)</code>. β > 1 weights <b>recall</b> more heavily.</li>
        <li>F2 is appropriate when <b>misses cost more than false alarms</b> — as here, where a missed failure costs about nineteen needless replacements.</li>
        <li>β &lt; 1 (F0.5) favours precision, appropriate when acting on a false alarm is itself harmful — an invasive medical follow-up, or removing a legitimate post.</li>
    </ul>
    </p>
</details>

**18. <code>classification_report</code> gives macro avg F1 of 0.820 and weighted avg F1 of 0.975 for the same model. Which should be reported, and why do they differ so much?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Macro</b> — or the minority class directly.</li>
        <li>The weighted average weights each class by its support, so on a fleet that is 96% healthy it is essentially the majority class talking, and it reads close to accuracy — inheriting the same blindness.</li>
        <li>Macro gives both classes equal weight, so the rare class registers.</li>
        <li>Reporting the weighted average is not incorrect, but it is the number that makes a mediocre model look finished. <b>Always say which average you used.</b></li>
    </ul>
    </p>
</details>

## Part 5 — The threshold

**19. Every metric above depended on the threshold 0.5. Where does that number come from, and what changes about the model as it moves?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>It is the default of <code>predict()</code> and nothing more. It is correct only when the two errors cost the same <b>and</b> the classes are balanced — neither holds here.</li>
        <li><b>Nothing about the model changes.</b> Same coefficients, same predicted probabilities, same drives. Only the line between "leave it" and "replace it" moves.</li>
        <li>So the useful response to "our classifier has 77% precision" is: <i>at what threshold, and why that one?</i></li>
    </ul>
    </p>
</details>

**20. Across the threshold sweep, accuracy ranged only from 0.942 to 0.977 while recall fell from 0.803 to 0.184, and accuracy peaked at 0.5. What does that imply about tuning on accuracy?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Accuracy cannot distinguish between models that differ enormously in what matters — a three-point spread covering a fourfold change in recall.</li>
        <li>Worse, its <b>maximum sits at a threshold that misses 33 of 76 failures</b>.</li>
        <li>So tuning on accuracy actively recommends <b>catching fewer failures</b>, because every alarm it drops was a possible false positive. It optimises in the wrong direction on an imbalanced problem.</li>
    </ul>
    </p>
</details>

**21. A false positive costs €140 and a false negative €2,600. Derive the cost-optimal threshold, and state what it does and does not depend on.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>For a drive with estimated probability p: flagging costs <code>(1-p)·C_FP</code> in expectation, leaving it costs <code>p·C_FN</code>. Flag when flagging is cheaper.</li>
        <li><code>(1-p)C_FP &lt; p·C_FN</code> gives <b>t* = C_FP / (C_FP + C_FN)</b> = 140/2740 = <b>0.051</b>.</li>
        <li>It depends <b>only on the ratio of the two costs</b> — not on the model, the dataset, or the class balance.</li>
        <li>If the errors cost the same, t* = 0.5 and the default was right. As misses grow costlier, t* falls towards zero.</li>
    </ul>
    </p>
</details>

**22. Choosing the threshold empirically gave 0.08, against a theoretical 0.051, and cut the cost from €87,620 to €45,540. Give one reason the discrepancy is not worrying, and one serious objection to the whole procedure.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Not worrying: the cost curve is <b>nearly flat around its minimum</b>, so many thresholds are almost equally good. That is fortunate, since cost estimates are never exact.</li>
        <li>The serious objection: <b>the threshold was chosen by looking at the test set</b>, which is exactly what lesson 1 forbids. The threshold is a parameter, and the test set has now influenced it.</li>
        <li>Done properly it is chosen on a validation set and measured once on test data. Until then the €45,540 is a demonstration of a method, not a reportable result.</li>
    </ul>
    </p>
</details>

## Part 6 — Ranking metrics and imbalance

**23. Define the two axes of the receiver operating characteristic (ROC) curve, and explain what the shape of the curve is actually measuring.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The <b>true positive rate</b>, TPR = TP/(TP+FN), which is recall — the failures caught. The <b>false positive rate</b>, FPR = FP/(FP+TN) — the healthy drives wrongly condemned.</li>
        <li>Both are 0 at threshold 1 and 1 at threshold 0. The entire content of the curve is the <b>order in which they get there</b>.</li>
        <li>A good model raises TPR fast while FPR is still low, because its highest scores really are the positives. A useless model raises both together and traces the diagonal.</li>
        <li>In words: as we grow more willing to raise alarms, do we catch failures faster than we annoy technicians?</li>
    </ul>
    </p>
</details>

**24. State the probabilistic meaning of the area under that curve (AUC), and describe how the lesson verified it without proving it.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>AUC is the probability that the model scores a randomly chosen positive above a randomly chosen negative.</b> It measures <b>ranking</b>.</li>
        <li>Notebook 3 drew 200,000 random (failing, healthy) pairs and counted how often the failing drive scored higher: <b>0.9488</b>, against <code>roc_auc_score</code>'s <b>0.9493</b>.</li>
        <li>The remaining gap is sampling noise and shrinks with more draws. The underlying identity is with the Mann-Whitney U statistic.</li>
    </ul>
    </p>
</details>

**25. Every predicted probability is divided by 10. What happens to the AUC, and what does that tell you about what AUC cannot check?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>The AUC is unchanged</b>, because dividing by 10 is monotone and preserves the ranking.</li>
        <li>Meanwhile the model is now badly <b>calibrated</b> and every threshold-based decision it makes is different.</li>
        <li>So AUC says nothing about whether "0.7" means anything. If you need believable probabilities — for example to compute expected costs — AUC is not the check you want.</li>
    </ul>
    </p>
</details>

**26. Thinning the failures from 3.8% to 0.4% of the fleet left AUC at 0.968 → 0.974 while precision at threshold 0.5 fell from 0.788 to 0.259. Explain the disagreement in terms of denominators.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>FPR divides by the number of healthy drives</b>, which is enormous and growing. A thousand false alarms among a hundred thousand healthy drives is an FPR of 0.01 — invisible on the ROC axis.</li>
        <li><b>Precision divides by the alarms raised</b>, where those same false alarms dominate.</li>
        <li>ROC measures false alarms against a population nobody experiences; the technician sees the alarm queue.</li>
        <li>The model was <b>not retrained</b> — its ranking ability is genuinely unchanged, which is why AUC is right to be stable. It is simply answering a different question. (The slight rise to 0.974 is sampling noise on 231 positives.)</li>
    </ul>
    </p>
</details>

**27. What is the baseline of a precision-recall curve, and why does that make it more informative than accuracy on an imbalanced problem?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>A model with random scores has precision equal to the <b>positive rate</b> at every recall — <b>0.038</b> here, not 0.5.</li>
        <li>So the floor of the plot moves with the difficulty of the problem, and any curve above it represents real skill.</li>
        <li>Our model's average precision was <b>0.717</b> against that floor of 0.038.</li>
        <li>The rule: on an imbalanced problem, report the precision-recall curve <b>alongside</b> AUC — never instead of it, and never AUC alone.</li>
    </ul>
    </p>
</details>

## Part 7 — Weights, resampling and multiclass

**28. <code>class_weight="balanced"</code> raised recall from 0.566 to 0.895 while AUC moved from 0.949 to 0.950. What does that pair of facts tell you about what class weighting does?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The AUC barely moved, so <b>the ranking is essentially unchanged</b> — reweighting taught the model nothing new about disk failure.</li>
        <li>What it changed is where the model puts its 0.5 line. It is <b>the same lever as the threshold</b>, pulled during training instead of after.</li>
        <li>So class weighting is not a <i>remedy</i> for imbalance; it is a reparameterisation of the same decision. The remedy, where one exists, is more positive examples.</li>
        <li>It earns its place when a downstream tool insists on <code>predict()</code> and gives no access to a threshold.</li>
    </ul>
    </p>
</details>

**29. Where in a cross-validation workflow must oversampling or SMOTE be applied, and what goes wrong otherwise?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Inside the fold</b>, after the split — never before it.</li>
        <li>Resampling before splitting places synthetic copies (or near-copies) of training examples into the validation set, so the model is scored on data derived from what it trained on.</li>
        <li>The result is an optimistic score that will not survive contact with new data — the same failure mode as fitting an imputer or scaler before splitting, in lesson 2.</li>
    </ul>
    </p>
</details>

**30. With three classes — healthy, degraded, failed — macro avg F1 was 0.707 and weighted avg 0.883. The degraded class scored precision 0.603 and recall 0.498. What should you conclude and report?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Degraded is the hard class</b>, and understandably so: it sits between two neighbours that both resemble it, so the model confuses it in both directions.</li>
        <li>Read the <b>off-diagonal cells</b> of the confusion matrix rather than the accuracy, since they say which confusions are being made.</li>
        <li>The macro–weighted gap of 0.18 is large because the healthy class is five drives in six.</li>
        <li>Which to report is a question about the application, not statistics: <b>if missing a failure matters as much as missing a healthy drive, macro is the honest one</b>. State which you used.</li>
    </ul>
    </p>
</details>

**31. The softmax generalises the sigmoid to K classes. State the relationship, and the cost function that goes with it.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><code>P(y = k | x) = exp(z_k) / Σ_j exp(z_j)</code> with one score z per class.</li>
        <li>For K = 2 it <b>reduces to the sigmoid</b>: subtract z₀ from both scores and one becomes 0, leaving σ(z₁ - z₀).</li>
        <li>The cost is <b>categorical cross-entropy</b>: the negative log of the probability the model assigned to the class that actually occurred — the same idea as the binary case.</li>
    </ul>
    </p>
</details>

## Part 8 — Putting it together

**32. A colleague reports: "Our churn classifier is 94% accurate with an AUC of 0.91." List the questions you would ask before believing it predicts churn usefully.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>What is the base rate?</b> If 94% of customers do not churn, the accuracy is exactly what predicting "no churn" for everyone achieves.</li>
        <li><b>What are precision and recall on the churn class</b>, and at what threshold? Accuracy and AUC both hide these.</li>
        <li><b>What is the average precision</b>, against a baseline equal to the positive rate? AUC alone hides imbalance.</li>
        <li><b>Was the threshold chosen on the test set?</b> And was any preprocessing, imputation or resampling fitted before the split?</li>
        <li><b>Macro or weighted average?</b></li>
        <li><b>What do the two errors cost?</b> Without that, no threshold can be defended.</li>
    </ul>
    </p>
</details>

**33. Summarise, in one sentence each, the two predictable mistakes this lesson was built around — and say why the reasoning behind each is defensible.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Trusting accuracy on an imbalanced problem.</b> Defensible because accuracy genuinely is the right summary on a balanced problem; it fails only when one class is rare, which is unfortunately most problems worth classifying.</li>
        <li><b>Reporting AUC alone.</b> Defensible because AUC is threshold-free, comparable across models and datasets, and bounded in a familiar range — all real virtues. It simply answers a question about ranking when the question was about the alarm queue.</li>
    </ul>
    </p>
</details>